# Model Evaluation - ChatKasir
- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## Import Library

In [1]:
import os
import re
import json
import numpy as np
import tensorflow as tf
from google.colab import drive
from tokenizers import Tokenizer
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, MultiHeadAttention

## Loading Assets

In [2]:
# Mount Google Drive
drive.mount('/content/drive')

# Definisikan Path
BASE_PATH = "/content/drive/MyDrive/ChatKasir/assets"
TOKENIZER_PATH = f"{BASE_PATH}/tokenizers/tokenizer.json"
CONFIG_PATH = f"{BASE_PATH}/data/model_config.json"
MODEL_PATH = f"{BASE_PATH}/models/chatkasir_model.keras"

# Muat Tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# uat Konfigurasi (Untuk mendapatkan Max Length)
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)
MAX_LENGTH = config['max_length']
print(f"Konfigurasi dimuat. Max Length: {MAX_LENGTH}")

Mounted at /content/drive
Konfigurasi dimuat. Max Length: 64


In [3]:
# Definisikan ulang Custom Layer (TransformerEncoder) agar bisa membangun kembali model
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.supports_masking = True
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32) if mask is not None else None
        attn_output = self.att(inputs, inputs, attention_mask=padding_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Muat Model AI
# compile=False karena hanya butuh model untuk menebak, bukan belajar.
# custom_objects memberi tahu Keras letak kelas TransformerEncoder.
print("Sedang memuat model AI...")
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'TransformerEncoder': TransformerEncoder},
    compile=False
)
print("Model AI 'Dual-Brain' berhasil dimuat dan siap digunakan")

Sedang memuat model AI...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_encoder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_encoder_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model AI 'Dual-Brain' berhasil dimuat dan siap digunakan


## Preprocessing Text to Matriks

In [23]:
def preprocess_input(text):
    # 1. Tokenisasi & Ubah teks menjadi ID angka
    encoded = tokenizer.encode(text)
    ids = encoded.ids

    # 2. Padding/Truncattng agar panjangnya tepat 64 (MAX_LENGTH)
    # Ini krusial karena arsitektur Transformer kita didesain untuk input ukuran tetap.
    if len(ids) > MAX_LENGTH:
        # Jika kepanjangan, potong bagian belakangnya
        ids = ids[:MAX_LENGTH]
    else:
        # Jika kependekan, tambahkan ID [PAD] (angka 0) di belakang
        selisih = MAX_LENGTH - len(ids)
        ids = ids + [0] * selisih

    # 3. Tambahkan dimensi batch
    # input bentuk (Batch, Max_Length) -> (1, 64)
    input_tensor = np.array([ids])

    return input_tensor

# --- TES FUNGSI PREPROCESSING ---
input_tes = "pak pesen 2 nasi goreng hrgnya brp sep harganya 15rb mas jadinya 30rb"
matriks_siap = preprocess_input(input_tes)

print(f"Kalimat: '{input_tes}'")
print(f"Bentuk Matriks: {matriks_siap.shape}")
print(f"Isi Matriks (10 angka pertama): {matriks_siap[0][:10]}")

Kalimat: 'pak pesen 2 nasi goreng hrgnya brp sep harganya 15rb mas jadinya 30rb'
Bentuk Matriks: (1, 64)
Isi Matriks (10 angka pertama): [307 175   6 107 145 649  48 740  59 212]


## Prediksi (Forward Pass)

In [24]:
print("AI sedang membaca dan berpikir...")

# Masukkan matriks (1, 64) ke dalam model
# Model akan mengembalikan list berisi 3 array sesuai urutan output di arsitektur
prediksi_mentah = model.predict(matriks_siap)

# Pisahkan ketiga output tersebut
raw_prod = prediksi_mentah[0]   # Output Cabang 1: Produk (NER)
raw_qty = prediksi_mentah[1]    # Output Cabang 2: Jumlah pesanan
raw_price = prediksi_mentah[2]  # Output Cabang 3: Harga satuan

print("\nPrediksi Selesai. Berikut adalah hasil mentah dari model AI:")
print("-" * 50)
print(f"1. Raw Product Shape : {raw_prod.shape}")
print("    > (1 kalimat, 64 kata, 3 probabilitas tag [O, B-PROD, I-PROD])")

print(f"\n2. Raw Quantity Value: {raw_qty[0][0]:.4f}")
print("    > Tebakan jumlah pesanan dalam bentuk desimal")

print(f"\n3. Raw Price Value   : {raw_price[0][0]:.4f}")
print("    > Tebakan harga satuan (masih dalam bentuk normalisasi)")
print("-" * 50)

AI sedang membaca dan berpikir...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step

Prediksi Selesai. Berikut adalah hasil mentah dari model AI:
--------------------------------------------------
1. Raw Product Shape : (1, 64, 3)
    > (1 kalimat, 64 kata, 3 probabilitas tag [O, B-PROD, I-PROD])

2. Raw Quantity Value: 2.0537
    > Tebakan jumlah pesanan dalam bentuk desimal

3. Raw Price Value   : 14.8916
    > Tebakan harga satuan (masih dalam bentuk normalisasi)
--------------------------------------------------


## Postprocessing

Di tahap ini, kita akan merakit logika untuk mengubah angka-angka raw tadi menjadi JSON yang rapi dan siap pakai. Logikanya meliputi:

1. Mencari probabilitas tertinggi dari matriks produk menggunakan np.argmax().
2. Mencocokkan tag produk (B-PROD dan I-PROD) dengan kamus Tokenizer untuk merangkai kembali kata aslinya.
3. Membulatkan hasil Quantity.
4. Mengembalikan nilai Price ke harga asli (dikali 1000) dan membulatkannya.

In [28]:
def postprocess_output(input_ids, raw_prod, raw_qty, raw_price, tokenizer, original_text):
    # 1. POSTPROCESSING PRODUK (NER) & SOFTMAX CONFIDENCE
    # Ambil tebakan dengan probabilitas tertinggi untuk setiap kata (Argmax)
    # Hasilnya berupa deretan angka 0 (O), 1 (B-PROD), atau 2 (I-PROD)
    probs = np.max(raw_prod[0], axis=-1)
    tag_ids = np.argmax(raw_prod[0], axis=-1)

    # Kumpulkan ID kata yang ditebak sebagai 1 (B-PROD) atau 2 (I-PROD)
    product_token_ids = []
    confidences = []
    for i, tag in enumerate(tag_ids):
        if tag in [1, 2]: # B-PROD atau I-PROD
            product_token_ids.append(input_ids[0][i])
            confidences.append(probs[i])

    # Ubah kembali deretan ID kata tersebut menjadi teks utuh
    nama_produk = tokenizer.decode(product_token_ids)
    # Hitung tingkat confidence
    avg_conf = np.mean(confidences) * 100 if confidences else 0
    conf_level = "HIGH" if avg_conf >= 90 else "MEDIUM" if avg_conf >= 70 else "LOW"
    print(f"DEBUG: Keyakinan AI (Softmax) untuk produk '{nama_produk}' adalah {avg_conf:.2f}%")

    # 2. POSTPROCESSING JUMLAH (QUANTITY)
    # Ambil angka desimalnya, lalu bulatkan menjadi bilangan bulat (Integer)
    qty_final = int(round(raw_qty[0][0]))

    # 3. POSTPROCESSING HARGA (PRICE)
    # Karena di training harga dibagi 1000, sekarang kita kalikan 1000
    # Lalu bulatkan menjadi bilangan bulat (Integer)
    price_mentah = raw_price[0][0] * 1000
    # Pembulatan harga
    price_final = int(round(price_mentah / 500.0) * 500)

    # Hitung total harga
    total_harga = qty_final * price_final

    # Pengecakan total harga berdasarkan teks chat
    # Sekarang bisa mendeteksi: "total 30rb", "jadinya 20000", "semuanya 50k"
    match = re.search(r'(?:total|jadi|semua)(?:nya)?\s*(\d+)\s*(rb|ribu|k)?', original_text, re.IGNORECASE)

    # 4. LOGIKA GABUNGAN CONFIDENCE (AI + BUSINESS RULE)
    if match:
        angka = int(match.group(1))
        satuan = match.group(2)
        total_chat = angka * 1000 if satuan and satuan.lower() in ['rb', 'ribu', 'k'] else angka

        # Jika total di chat tidak sama dengan prediksi model -> Langsung LOW
        if total_chat != total_harga:
            conf_level = "LOW"
        else:
            # Karena total harga dari chat cocok 100% dengan total harga
            # hasil perkalian (qty * price_satuan), maka abaikan keraguan
            # AI pada nama produk dan langsung berikan status HIGH.
            conf_level = "HIGH"
    else:
        # Jika tidak ada kata "total/jadi/semua" di chat, kita gunakan murni keyakinan AI
        conf_level = "HIGH" if avg_conf >= 90 else "MEDIUM" if avg_conf >= 70 else "LOW"

    # 5. BUNGKUS KE DALAM JSON
    response = {
        "results": [
            {
                "product": nama_produk,
                "quantity": qty_final,
                "price_satuan": price_final,
                "total": total_harga,
                "confidence": conf_level
            }
        ],
        "clean_text": original_text
    }

    return response

# --- TES POSTPROCESSING ---
hasil_json = postprocess_output(matriks_siap, raw_prod, raw_qty, raw_price, tokenizer, input_tes)

print("\nHASIL AKHIR INFERENCE (SIAP KIRIM KE DATABASE):")
print("-" * 50)
print(json.dumps(hasil_json, indent=4))
print("-" * 50)

DEBUG: Keyakinan AI (Softmax) untuk produk 'nasi goreng' adalah 41.51%

HASIL AKHIR INFERENCE (SIAP KIRIM KE DATABASE):
--------------------------------------------------
{
    "results": [
        {
            "product": "nasi goreng",
            "quantity": 2,
            "price_satuan": 15000,
            "total": 30000,
            "confidence": "HIGH"
        }
    ],
    "clean_text": "pak pesen 2 nasi goreng hrgnya brp sep harganya 15rb mas jadinya 30rb"
}
--------------------------------------------------
